In [ ]:
from typing import Mapping, Sequence
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

import pulp

In [ ]:
# Constants

INPUT_BASE_PATH = Path("../data")
FORECASTS_PATH = Path("../results/ets/submissions")

FOODS_PRODUCT_IDS = [
    "FOODS_3_555_TX_2",
    "FOODS_3_376_TX_2",
    "FOODS_3_811_CA_2",
    "FOODS_1_218_TX_1",
    "FOODS_3_226_WI_3",
    "FOODS_3_070_WI_2",
    "FOODS_3_007_TX_2",
    "FOODS_3_444_TX_1",
    "FOODS_2_398_WI_3",
    "FOODS_3_540_WI_1",
]
HOUSEHOLD_PRODUCT_IDS = [
    "HOUSEHOLD_1_334_TX_1",
    "HOUSEHOLD_1_459_CA_2",
    "HOUSEHOLD_2_342_WI_2",
    "HOUSEHOLD_1_465_TX_3",
    "HOUSEHOLD_1_294_WI_2",
    "HOUSEHOLD_2_176_CA_3",
    "HOUSEHOLD_1_334_TX_2",
    "HOUSEHOLD_1_106_WI_2",
    "HOUSEHOLD_1_474_TX_2",
    "HOUSEHOLD_1_106_TX_1",
]
HOBBIES_PRODUCT_IDS = [
    "HOBBIES_1_048_WI_1",
    "HOBBIES_1_067_CA_3",
    "HOBBIES_1_158_TX_3",
    "HOBBIES_1_404_WI_3",
    "HOBBIES_1_234_CA_3",
    "HOBBIES_1_254_CA_3",
    "HOBBIES_1_019_WI_1",
    "HOBBIES_1_370_WI_1",
    "HOBBIES_1_048_CA_1",
    "HOBBIES_1_354_TX_3",
]
ALL_PRODUCT_IDS = list(FOODS_PRODUCT_IDS + HOUSEHOLD_PRODUCT_IDS + HOBBIES_PRODUCT_IDS)

# ==================
# FORECAST CONSTANTS 
# ==================
MAX_TRAINING_TIMESTAMP = 1913
FORECAST_HORIZON = 7
N_FORECAST_TIMESTAMPS = 4

# Timestamps at which we will generate forecasts. We will assume that we have already
# observed the ground truth value of each product series at the forecast timestamp and
# we produce forcasts for the following FORECAST_HORIZON timestamps.
FORECAST_TIMESTAMPS = [MAX_TRAINING_TIMESTAMP + i * FORECAST_HORIZON for i in range(N_FORECAST_TIMESTAMPS)]
FORECASTED_TIMESTAMPS = [list(range(t + 1, t + 1 + FORECAST_HORIZON)) for t in FORECAST_TIMESTAMPS]

In [ ]:
# Load data

CALENDAR_DATA = pl.read_csv(f"{INPUT_BASE_PATH}/calendar.csv", try_parse_dates=True)
SELL_PRICES = pl.read_csv(f"{INPUT_BASE_PATH}/sell_prices.csv")
SALES_TRAIN_VALIDATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_validation.csv")
SALES_TRAIN_EVALUATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_evaluation.csv")

In [ ]:
# Baseline forecasts

class M5HistoricalAverageModel:
    def __init__(self, lookback: int = 1, horizon: int = 1):
        self.lookback = lookback
        self.horizon = horizon

        self._product_ids: tuple[str, ...] | None = None
        self._max_train_index_by_product: dict[str, int] | None = None

    @property
    def product_ids(self) -> tuple[str, ...] | None:
        return self._product_ids

    def fit(self, train_df: pl.DataFrame) -> "M5HistoricalAverageModel":
        # Get unique product ids
        self._product_ids = tuple(train_df["id"].unique())

        # Calculate min/max timestamps for each product.
        start_end_index = (
            train_df
            .group_by(["id"])
            .agg(t_end=pl.col("d_index").max())
            .with_columns(t_start=pl.col("t_end") - self.lookback)
        )
        self._max_train_index_by_product = {
            d["id"]: d["t_end"] 
            for d in start_end_index[["id", "t_end"]].to_dicts()
        }

        # Join back onto train_df, filter to index within range
        # and calculate average
        avg_sales = (
            train_df
            .join(start_end_index, on="id", how="inner")
            .filter(pl.col("d_index").is_between(pl.col("t_start"), pl.col("t_end"), closed="both"))
            .group_by(["id"])
            .agg(avg_sales=pl.col("sales").mean())
        )
        self._avg_sales_by_product = {
            d["id"]: d["avg_sales"] 
            for d in avg_sales[["id", "avg_sales"]].to_dicts()
        }
        
        return self

    def predict(self, horizon: int | None = None) -> pl.DataFrame:
        if horizon is None:
            horizon = self.horizon
        elif isinstance(horizon, int) and horizon < 1:
            raise ValueError(f"Horizon has to be > 1. Got {horizon=}")
        
        forecast_dfs: list[pl.DataFrame] = []
        for product_id in self.product_ids:
            max_t_for_product = self._max_train_index_by_product[product_id]
            avg_sales_for_product = self._avg_sales_by_product[product_id]
            product_forecast_df = pl.DataFrame(
                {
                    "id": [product_id for _ in range(horizon)],
                    "F_index": [max_t_for_product + i for i in range(1, horizon + 1)],
                    "sales": [avg_sales_for_product for _ in range(horizon)]
                }
            )
            forecast_dfs.append(product_forecast_df)

        return pl.concat(forecast_dfs)



In [ ]:
SALES_DF = (
    SALES_TRAIN_EVALUATION
    .with_columns(id=pl.col("id").str.strip_suffix("_evaluation"))
    .filter(pl.col("id").is_in(ALL_PRODUCT_IDS))
    .unpivot(
        index=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
        variable_name="d",
        value_name="sales"
    ).with_columns(
        d_index=pl.col("d").str.extract(r"^d_([0-9]+)").cast(pl.Int64)
    )
)

SALES_DF.head()

In [ ]:
# Generate forecasts at each forecast timestamp.
# The ground truth sales observations are observed at each forecast timestamp
# (i.e. they are included in the train data) and forecasts are generated
# for each timestamp in the forecast horizon after the forecast timestamp.

forecast_timestamps = [MAX_TRAINING_TIMESTAMP + i for i in range(2)]

for forecast_timestamp in forecast_timestamps:

    train_df = SALES_DF.filter(
        pl.col("d_index") <= forecast_timestamp
    )
    valid_df = SALES_DF.filter(
        pl.col("d_index") >= forecast_timestamp + 1,
        pl.col("d_index") <= forecast_timestamp + FORECAST_HORIZON,
    )
    
    
    model = M5HistoricalAverageModel(lookback=0, horizon=7)
    model.fit(train_df)
    forecast_df = model.predict()

    break

In [ ]:
product_id = SALES_DF["id"].sample(1).item()

fig, ax = plt.subplots()

train_df = SALES_DF.filter(
    pl.col("id") == product_id,
    pl.col("d_index").is_between((MAX_TRAINING_TIMESTAMP - 7), MAX_TRAINING_TIMESTAMP)
).sort(by="d_index")

ax.plot(
    train_df["d_index"].to_list(),
    train_df["sales"].to_list(),
)
# ax.axhline(train_df["sales"].mean())

product_forecast_df = forecast_df.filter(pl.col('id') == product_id).sort(by="F_index")
ax.plot(
    product_forecast_df["F_index"].to_list(),
    product_forecast_df["sales"].to_list(),
)

product_valid_df = valid_df.filter(pl.col('id') == product_id).sort(by="d_index")
ax.plot(
    product_valid_df["d_index"].to_list(),
    product_valid_df["sales"].to_list(),
)

In [ ]:
SALES_DF["id"].sample(2)

In [ ]:
timesteps = range(FORECAST_HORIZON)
gamma = 0.95
product_ids = ["HOUSEHOLD_1_106_TX_1", "FOODS_3_376_TX_2"]

# Get forecasts by product id
product_demand_forecasts: dict[str, list[float]] = {}
for product_id in product_ids:
    product_forecast_df = forecast_df.filter(pl.col("id")==product_id).sort(by='F_index')
    product_demand_forecasts[product_id] = product_forecast_df["sales"].to_list()


# Get sales prices by product id
product_sale_prices: dict[str, list[float]] = {}
for product_id in product_ids:
    # TODO: Replace with actual sales prices
    product_sale_prices[product_id] = [9.5 for _ in range(FORECAST_HORIZON)]


# Get storage costs by product id
product_storage_prices: dict[str, list[float]] = {}
for product_id in product_ids:
    # TODO: Replace with actual storage prices
    product_storage_prices[product_id] = [3.5 for _ in range(FORECAST_HORIZON)]


# Get storage volume by product id
product_storage_volumes: dict[str, float] = {}
for product_id in product_ids:
    # TODO: Replace with actual storage prices
    product_storage_volumes[product_id] = 12.5

# Define LP Problem

problem = pulp.LpProblem("mpc", sense=pulp.LpMinimize)

stock_addition_var = pulp.LpVariable.dicts("stock_addition", indices=(product_ids, timesteps), lowBound=0, cat=pulp.LpInteger)
stock_var = pulp.LpVariable.dicts("stock", indices=(product_ids, timesteps), lowBound=0, cat=pulp.LpInteger)
auxiliary_var = pulp.LpVariable.dicts("t", indices=(product_ids, timesteps), lowBound=0, cat=pulp.LpInteger)

# Objective function
problem += (
    pulp.lpSum(
        gamma ** k * auxiliary_var[p][k]
        for k in timesteps
        for p in product_ids
    )
)

# Constraints
for k in timesteps:
    for p in product_ids:

        # Constraint for missing out on possible sales (stocking less than demand)
        problem += (
            product_sale_prices[p][k] * (product_demand_forecasts[p][k] - stock_var[p][k] - stock_addition_var[p][k])
        ) <= auxiliary_var[p][k]

        # Constraint for storing more than expected demand
        problem += (
            product_storage_prices[p][k] * (stock_var[p][k] + stock_addition_var[p][k] - product_demand_forecasts[p][k])
        ) <= auxiliary_var[p][k]


# TODO: Constraint on initial stock levels.


# Storage capacity constraint
for k in timesteps:
    # TODO: Replace with actual storage capacity
    problem += pulp.lpSum(
        [
            (stock_var[p][k] + stock_addition_var[p][k]) *  product_storage_volumes[p]
            for p in product_ids
        ]
    ) <= 10_000


# Transition model

In [ ]:
problem